# Foundations of Data Manipulation and Data Analysis
## Global Missing Migrants Dataset

This notebook provides a practical introduction to data analysis using the **Global Missing Migrants Dataset**.

### Learning objectives
- Understand Python data structures and Pandas Series/DataFrames.
- Inspect and clean a real-world CSV dataset.
- Identify data types, missing values, duplicates, and data-quality issues.
- Apply descriptive statistics and grouped summaries.
- Explore patterns by region, route, cause, location, and date where those fields are available.
- Create appropriate visualizations and interpret results cautiously.
- Export cleaned data and summary tables.

**Important:** Records of missing migrants and deaths/disappearances are documented cases and may not represent the full scale of migration-related deaths or disappearances. Treat the data as incomplete and avoid interpreting missing records as evidence that an event did not occur.

Run the notebook from top to bottom.

## 1. Setup and imports

We use `pandas` for data manipulation, `numpy` for numerical operations, `matplotlib` and `seaborn` for visualizations, and `pathlib` for file paths.

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Do not hide warnings while learning: warnings can reveal data or parsing issues.
sns.set_theme()
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 40)

# Keep the CSV in the same folder as this notebook whenever possible.
candidate_paths = [
    Path("Global Missing Migrants Dataset.csv"),
    Path("Global_Missing_Migrants_Dataset.csv"),
    Path("/mnt/data/Global Missing Migrants Dataset.csv"),
    Path("/mnt/data/Global_Missing_Migrants_Dataset.csv"),
]

FILE_PATH = next((p for p in candidate_paths if p.exists()), None)

# If the exact filename differs, look for a likely matching CSV in the current folder.
if FILE_PATH is None:
    likely_csvs = [
        p for p in Path.cwd().glob("*.csv")
        if "missing" in p.name.lower() and "migrant" in p.name.lower()
    ]
    if len(likely_csvs) == 1:
        FILE_PATH = likely_csvs[0]

if FILE_PATH is None:
    raise FileNotFoundError(
        "Dataset CSV not found. Place 'Global Missing Migrants Dataset.csv' "
        "in the same folder as this notebook, or update FILE_PATH to the correct path."
    )

print("Dataset file:", FILE_PATH.resolve())


## 2. Load the CSV file

A CSV file is a plain-text table. `read_csv()` loads it into a Pandas DataFrame. Keep the source CSV in the same folder as this notebook, or update `FILE_PATH` in the setup cell. We first inspect the dimensions and a few records.


In [ ]:
df_raw = pd.read_csv(FILE_PATH, low_memory=False)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")
display(df_raw.head())

## 3. Foundational data structures

- **DataFrame:** a two-dimensional table.
- **Series:** one column from a DataFrame.
- **List:** an ordered collection such as the column names.
- **Dictionary:** key-value pairs useful for metadata and summaries.
- **Data types:** indicate whether a column contains text, numbers, dates, or Boolean values.

In [ ]:
column_names = df_raw.columns.tolist()
dataset_metadata = {
    "dataset_name": "Global Missing Migrants Dataset",
    "rows": int(df_raw.shape[0]),
    "columns": int(df_raw.shape[1]),
    "column_names": column_names
}

print("DataFrame type:", type(df_raw))
print("Series type:", type(df_raw.iloc[:, 0]))
print("Column-name list type:", type(column_names))
print("Metadata dictionary type:", type(dataset_metadata))
print("\nColumn names:")
print(column_names)

## 4. Data dictionary and first inspection

This table summarizes column types, missing counts, and missing percentages. The dataset's exact column names may evolve between published versions, so later sections identify likely fields from their names.

In [ ]:
data_dictionary = pd.DataFrame({
    "column": df_raw.columns.astype(str),
    "dtype": df_raw.dtypes.astype(str).values,
    "missing_count": df_raw.isna().sum().values,
    "missing_percent": (df_raw.isna().mean().values * 100).round(2),
    "unique_values": df_raw.nunique(dropna=True).values
}).sort_values("missing_percent", ascending=False)

display(data_dictionary)

In [ ]:
display(df_raw.describe(include="all").T)

In [ ]:
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))

## 5. Standardize column names and tidy blank values

We create a working copy and standardize column names for easier coding. The original DataFrame `df_raw` remains unchanged. Empty strings and strings containing only whitespace are treated as missing values. Values such as `"Unknown"` are **not** automatically replaced because they may be meaningful categories in the source.


In [ ]:
df = df_raw.copy()

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
)

# Normalize whitespace in text columns and convert blank strings to missing values.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].map(lambda value: value.strip() if isinstance(value, str) else value)
    df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)

print("Standardized column names:")
print(df.columns.tolist())


In [ ]:
display(df.head())

## 6. Data quality: missing values and duplicates

Missing information is common in incident-level humanitarian datasets. A missing value is not the same as zero. We report exact duplicate rows, but do not automatically remove them by default: two rows can look identical while representing distinct events. Change `REMOVE_EXACT_DUPLICATES` to `True` only after checking the source documentation and deciding that exact duplicates should be excluded.


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

In [ ]:
exact_duplicate_count = int(df.duplicated().sum())
REMOVE_EXACT_DUPLICATES = False  # Change only after reviewing the source and duplicate rows.

if REMOVE_EXACT_DUPLICATES:
    df_clean = df.drop_duplicates().copy()
else:
    df_clean = df.copy()

print(f"Rows in working data: {len(df):,}")
print(f"Exact duplicate rows detected: {exact_duplicate_count:,}")
print(f"Rows used for analysis: {len(df_clean):,}")
print(f"Automatic duplicate removal enabled: {REMOVE_EXACT_DUPLICATES}")


## 7. Identify date, year, numeric, and categorical columns

A year such as `2014` is a year value, not a complete timestamp. Passing the integer `2014` directly to `pandas.to_datetime()` can interpret it as nanoseconds after 1 January 1970 and create a misleading trend. This section parses only columns that look like actual dates, while detecting year and month fields separately.


In [ ]:
# Only columns explicitly named as dates are parsed as full dates.
date_candidates = [
    c for c in df_clean.columns
    if "date" in c.lower() and "update" not in c.lower()
]

date_parse_report = []
for col in date_candidates:
    parsed = pd.to_datetime(df_clean[col], errors="coerce")
    nonmissing = int(df_clean[col].notna().sum())
    parse_rate = parsed.notna().sum() / nonmissing if nonmissing else 0
    date_parse_report.append({
        "column": col,
        "parse_rate_percent": round(parse_rate * 100, 2)
    })
    if nonmissing and parse_rate >= 0.70:
        df_clean[col] = parsed

display(pd.DataFrame(date_parse_report, columns=["column", "parse_rate_percent"]))

# Detect year and month columns independently from full dates.
year_candidates = [c for c in df_clean.columns if "year" in c.lower()]
month_candidates = [c for c in df_clean.columns if "month" in c.lower()]

YEAR_COL = year_candidates[0] if year_candidates else None
MONTH_COL = month_candidates[0] if month_candidates else None

numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
date_cols = df_clean.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Full date columns:", date_cols)
print("Year column:", YEAR_COL)
print("Month column:", MONTH_COL)
print("Categorical columns:", categorical_cols)


In [ ]:
print("Numeric columns:", numeric_cols)
print("Date columns:", date_cols)
print("Categorical columns:", categorical_cols)

## 8. Explore category frequencies

Frequency tables help identify the most common values in fields such as region, route, cause, or location. The notebook uses the actual categorical columns available in the uploaded dataset.

In [ ]:
for col in categorical_cols[:10]:
    print(f"\nTop values for: {col}")
    display(
        df_clean[col].astype("string")
        .fillna("(Missing)")
        .value_counts()
        .head(15)
        .rename_axis(col)
        .reset_index(name="record_count")
    )

## 9. Descriptive statistics

For numeric variables, calculate count, mean, standard deviation, minimum, quartiles, and maximum. Interpret these summaries according to each variable's meaning and units.

In [ ]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    display(df_clean[numeric_cols].describe().T)
else:
    print("No numeric columns were detected. Inspect the data dictionary and convert numeric-looking fields where appropriate.")

## 10. Detect likely analytical dimensions

Column names can differ across dataset releases. The selection below prioritizes specific fields such as `cause_of_death` and `country_of_origin` instead of accidentally selecting `incident_type` or `region_of_origin` because their names contain a broad keyword. Review the selected columns before interpreting results.


In [ ]:
def first_matching_column(columns, preferred_names=(), keywords=()):
    """Return the first exact preferred name found, then a keyword match."""
    for name in preferred_names:
        if name in columns:
            return name
    for col in columns:
        if any(keyword in col.lower() for keyword in keywords):
            return col
    return None

all_cols = df_clean.columns.tolist()

DATE_COL = first_matching_column(
    all_cols,
    preferred_names=("incident_date", "date_of_incident", "reported_date"),
    keywords=("date",)
)
YEAR_COL = first_matching_column(all_cols, preferred_names=("incident_year", "year"), keywords=("year",))
MONTH_COL = first_matching_column(all_cols, preferred_names=("reported_month", "incident_month", "month"), keywords=("month",))
REGION_COL = first_matching_column(
    all_cols,
    preferred_names=("region_of_incident", "region", "region_of_origin"),
    keywords=("region", "subregion")
)
ROUTE_COL = first_matching_column(
    all_cols,
    preferred_names=("migration_route", "route"),
    keywords=("route",)
)
CAUSE_COL = first_matching_column(
    all_cols,
    preferred_names=("cause_of_death", "cause"),
    keywords=("cause_of_death", "cause")
)
COUNTRY_COL = first_matching_column(
    all_cols,
    preferred_names=("country_of_incident", "country_of_origin", "country"),
    keywords=("country",)
)
LOCATION_COL = first_matching_column(
    all_cols,
    preferred_names=("location_of_death", "location"),
    keywords=("location",)
)
TOTAL_COUNT_COL = first_matching_column(
    all_cols,
    preferred_names=("total_number_of_dead_and_missing", "number_of_dead", "total_dead_and_missing"),
    keywords=("total_number_of_dead_and_missing", "number_of_dead_and_missing", "number_of_dead", "total_dead")
)

print("Selected fields:")
print({
    "date": DATE_COL,
    "year": YEAR_COL,
    "month": MONTH_COL,
    "region": REGION_COL,
    "route": ROUTE_COL,
    "cause_of_death": CAUSE_COL,
    "country": COUNTRY_COL,
    "location": LOCATION_COL,
    "primary count measure": TOTAL_COUNT_COL,
})


## 11. Visualize categorical patterns

Bar charts are useful for comparing counts across categories. Here we plot the most frequently recorded categories in selected dimensions when those fields are present.

In [ ]:
plot_columns = [c for c in [REGION_COL, ROUTE_COL, CAUSE_COL, COUNTRY_COL] if c]
for col in plot_columns[:4]:
    counts = (
        df_clean[col].astype("string").fillna("(Missing)")
        .value_counts().head(15).sort_values()
    )
    counts.plot(kind="barh")
    plt.title(f"Top recorded categories: {col}")
    plt.xlabel("Number of records")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

## 12. Date-based analysis and trends

Use a full date field if one is available. Otherwise, use the year column as a numeric year—not as a timestamp. These charts show the number of **records in the dataset** by year. They do not directly measure the true number of migration-related deaths or disappearances in the world.


In [ ]:
# Prefer a true date field; otherwise, analyze a numeric year field.
if DATE_COL:
    dated = df_clean.copy()
    dated["analysis_date"] = pd.to_datetime(dated[DATE_COL], errors="coerce")
    dated = dated.dropna(subset=["analysis_date"])
    if not dated.empty:
        dated["analysis_year"] = dated["analysis_date"].dt.year
        annual_records = dated.groupby("analysis_year").size().rename("record_count")
        display(annual_records.to_frame())
        annual_records.plot(marker="o")
        plt.title("Documented records by year")
        plt.xlabel("Year")
        plt.ylabel("Number of records")
        plt.tight_layout()
        plt.show()
    else:
        print("A date column was detected, but no values could be parsed.")
elif YEAR_COL:
    year_values = pd.to_numeric(df_clean[YEAR_COL], errors="coerce")
    # Exclude implausible values rather than silently plotting them.
    valid_year = year_values.between(1900, 2100)
    dated = df_clean.loc[valid_year].copy()
    dated["analysis_year"] = year_values.loc[valid_year].astype(int)
    if not dated.empty:
        annual_records = dated.groupby("analysis_year").size().rename("record_count")
        display(annual_records.to_frame())
        annual_records.plot(marker="o")
        plt.title("Documented records by incident year")
        plt.xlabel("Incident year")
        plt.ylabel("Number of records")
        plt.tight_layout()
        plt.show()

        if MONTH_COL:
            month_map = {
                "january": 1, "jan": 1, "february": 2, "feb": 2,
                "march": 3, "mar": 3, "april": 4, "apr": 4,
                "may": 5, "june": 6, "jun": 6, "july": 7, "jul": 7,
                "august": 8, "aug": 8, "september": 9, "sep": 9, "sept": 9,
                "october": 10, "oct": 10, "november": 11, "nov": 11,
                "december": 12, "dec": 12
            }
            month_num = df_clean.loc[valid_year, MONTH_COL].astype("string").str.strip().str.lower().map(month_map)
            monthly = pd.DataFrame({
                "year": year_values.loc[valid_year].astype(int),
                "month_num": month_num
            }).dropna()
            if not monthly.empty:
                monthly["year_month"] = (
                    monthly["year"].astype(int).astype(str) + "-" +
                    monthly["month_num"].astype(int).astype(str).str.zfill(2)
                )
                monthly_records = monthly.groupby("year_month").size().rename("record_count")
                monthly_records.tail(60).plot()
                plt.title("Documented records by month (latest 60 year-months)")
                plt.xlabel("Year-month")
                plt.ylabel("Number of records")
                plt.xticks(rotation=45, ha="right")
                plt.tight_layout()
                plt.show()
    else:
        print("No plausible year values were found in the detected year column.")
else:
    print("No full date or year column was detected. Review the dataset columns and set DATE_COL or YEAR_COL manually.")


## 13. Numeric distributions and outliers

Histograms show the distribution of numeric variables. Box plots help reveal extreme values, which should be investigated rather than removed automatically. Incident counts may be highly skewed.

In [ ]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

for col in numeric_cols[:4]:
    values = pd.to_numeric(df_clean[col], errors="coerce").dropna()
    if values.empty:
        continue

    plt.figure(figsize=(9, 4))
    plt.hist(values, bins="auto", edgecolor="black")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 2.5))
    plt.boxplot(values, vert=False)
    plt.title(f"Box plot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


## 14. Correlation analysis

Correlation summarizes linear association between numeric variables; it does not establish causation. For this dataset, counts such as total dead and missing, number dead, and estimated missing may be mathematically related. A high correlation between these fields may therefore reflect how the measures are constructed, not an independent causal relationship. We focus on numeric fields that appear to describe people or incident counts, and exclude year-like fields from the heatmap.


In [ ]:
# Select numeric measures related to incident or people counts; exclude year and ID-like fields.
count_keywords = ("number", "total", "dead", "missing", "surviv", "female", "male", "child")
candidate_numeric_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
metric_cols = [
    col for col in candidate_numeric_cols
    if any(keyword in col.lower() for keyword in count_keywords)
    and "year" not in col.lower()
    and not col.lower().endswith("_id")
]

if len(metric_cols) >= 2:
    # Remove constant columns and columns with too few valid observations.
    variable_cols = [
        col for col in metric_cols
        if df_clean[col].nunique(dropna=True) > 1 and df_clean[col].notna().sum() >= 3
    ]
    if len(variable_cols) >= 2:
        corr = df_clean[variable_cols].corr(method="pearson", min_periods=3)
        display(corr.round(2))

        corr_view = corr.iloc[:15, :15]
        plt.figure(figsize=(12, 8))
        sns.heatmap(
            corr_view, cmap="vlag", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", linewidths=0.5,
            cbar_kws={"label": "Pearson correlation coefficient"}
        )
        plt.title("Correlation among recorded count variables")
        plt.xticks(rotation=45, ha="right")
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

        upper_triangle = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        strong_corr = upper_triangle.stack().rename("correlation").reset_index()
        strong_corr.columns = ["variable_1", "variable_2", "correlation"]
        strong_corr["absolute_correlation"] = strong_corr["correlation"].abs()
        strong_corr = strong_corr.sort_values("absolute_correlation", ascending=False)
        print("Strongest observed correlations (inspect variable definitions before interpreting):")
        display(strong_corr.head(10).round(3))
    else:
        print("At least two numeric count variables with variation and enough valid values are required.")
else:
    print("No suitable numeric count variables were detected for correlation analysis.")


## 15. Statistical methods: mean, median, spread, and percentiles

These measures describe the distribution of numeric variables. For count data, distributions can be skewed by a small number of incidents involving many people. Compare the mean with the median and inspect the quartiles rather than relying on the mean alone. The summary is descriptive and does not by itself establish population-level risk.


In [ ]:
def descriptive_summary(data, column):
    values = pd.to_numeric(data[column], errors="coerce").dropna()
    return {
        "variable": column,
        "valid_n": int(values.size),
        "mean": values.mean(),
        "median": values.median(),
        "standard_deviation": values.std(),
        "variance": values.var(),
        "minimum": values.min(),
        "25th_percentile": values.quantile(0.25),
        "75th_percentile": values.quantile(0.75),
        "maximum": values.max()
    }

summary_cols = [
    col for col in df_clean.select_dtypes(include=np.number).columns
    if col != YEAR_COL and not col.lower().endswith("_id")
]
if summary_cols:
    display(pd.DataFrame([descriptive_summary(df_clean, c) for c in summary_cols[:10]]))
else:
    print("No numeric measurement columns are available for descriptive statistics.")


## 16. Grouped summaries

`groupby()` compares the number of documented records across regions, routes, causes, or countries. If numeric count fields are summarized, interpret them according to their definitions and missingness. A region with more records is not automatically a region with greater underlying risk.


In [ ]:
for group_col in [REGION_COL, ROUTE_COL, CAUSE_COL, COUNTRY_COL]:
    if group_col:
        print(f"\nDocumented record counts by {group_col}")
        display(
            df_clean.groupby(group_col, dropna=False)
            .size().sort_values(ascending=False)
            .head(20).rename("record_count").to_frame()
        )

if REGION_COL and summary_cols:
    value_cols = summary_cols[:5]
    print(f"\nNumeric summaries by {REGION_COL}")
    display(
        df_clean.groupby(REGION_COL, dropna=False)[value_cols]
        .agg(["count", "mean", "median", "max"])
        .head(20)
    )


## 17. Optional geospatial exploration

The dataset may store coordinates as one text field such as `"31.650259, -110.366453"` rather than separate latitude and longitude columns. The code below supports both formats, checks coordinate ranges, and plots valid recorded locations. Coordinates may be approximate and should not be treated as precise locations of individuals.


In [ ]:
lat_candidates = [c for c in df_clean.columns if c in ["lat", "latitude"] or c.endswith("_latitude")]
lon_candidates = [c for c in df_clean.columns if c in ["lon", "lng", "longitude"] or c.endswith("_longitude")]
coordinate_candidates = [c for c in df_clean.columns if "coordinate" in c.lower()]

geo = None
if lat_candidates and lon_candidates:
    lat_col, lon_col = lat_candidates[0], lon_candidates[0]
    geo = df_clean[[lat_col, lon_col]].copy()
    geo["latitude_plot"] = pd.to_numeric(geo[lat_col], errors="coerce")
    geo["longitude_plot"] = pd.to_numeric(geo[lon_col], errors="coerce")
elif coordinate_candidates:
    coordinate_col = coordinate_candidates[0]
    split_coords = df_clean[coordinate_col].astype("string").str.split(",", n=1, expand=True)
    if split_coords.shape[1] == 2:
        geo = pd.DataFrame(index=df_clean.index)
        geo["latitude_plot"] = pd.to_numeric(split_coords[0].str.strip(), errors="coerce")
        geo["longitude_plot"] = pd.to_numeric(split_coords[1].str.strip(), errors="coerce")

if geo is not None:
    geo = geo.dropna(subset=["latitude_plot", "longitude_plot"])
    geo = geo[
        geo["latitude_plot"].between(-90, 90)
        & geo["longitude_plot"].between(-180, 180)
    ]
    valid_coordinate_count = len(geo)
    print(f"Records with valid coordinate pairs: {valid_coordinate_count:,}")
    if len(geo) > 10000:
        geo = geo.sample(10000, random_state=42)
        print("Plotting a reproducible sample of 10,000 valid coordinate pairs.")
    if not geo.empty:
        plt.figure(figsize=(10, 6))
        plt.scatter(geo["longitude_plot"], geo["latitude_plot"], alpha=0.35, s=10)
        plt.title("Recorded incident coordinates (not a population-risk map)")
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.tight_layout()
        plt.show()
    else:
        print("Coordinate fields were found, but no valid coordinate pairs remain.")
else:
    print("No separate latitude/longitude fields or combined coordinate field were detected.")


## 18. Export cleaned data and summaries

Exports are written to an `analysis_outputs` folder next to the notebook's current working directory. Review the cleaning decisions and outputs before sharing. Because exact duplicate removal is disabled by default, the exported file preserves identical rows unless you explicitly enable that option above.


In [ ]:
from pathlib import Path

output_dir = Path("analysis_outputs")
output_dir.mkdir(exist_ok=True)

cleaned_output = output_dir / "global_missing_migrants_cleaned.csv"
missing_output = output_dir / "global_missing_migrants_missing_summary.csv"
dictionary_output = output_dir / "global_missing_migrants_data_dictionary.csv"

df_clean.to_csv(cleaned_output, index=False)
missing_summary.to_csv(missing_output)
data_dictionary.to_csv(dictionary_output, index=False)

print("Saved files:")
print(cleaned_output.resolve())
print(missing_output.resolve())
print(dictionary_output.resolve())

## 19. Evidence-based findings checklist\n\nUse the outputs above to write findings only after running the notebook on the current CSV. Do not invent results or copy old figures from a previous run. Complete this checklist in your project report:\n\n1. **Dataset scope:** report the number of rows and columns and explain what one row represents, based on the dataset documentation.\n2. **Data quality:** name the columns with the most missing values and report how many exact duplicate rows were detected. State whether duplicates were removed.\n3. **Temporal pattern:** identify years with the highest and lowest documented record counts, if the year field is usable. State that changes may reflect reporting coverage as well as real-world changes.\n4. **Category pattern:** identify the most frequent recorded region, route, or cause, and specify which field was counted.\n5. **Count variables:** describe one numeric measure using its valid observation count, median, and interquartile range. Explain whether the distribution appears skewed.\n6. **Limitations:** explain missingness, possible underreporting, category ambiguity, duplicate uncertainty, and why documented counts are not population-level risk estimates.\n\n**Suggested wording:** “The analysis describes patterns in recorded incidents in the dataset. It does not estimate the full number of deaths or disappearances, and differences between regions may reflect differences in exposure, reporting, and documentation.”\n

## 20. Practice exercises

1. Identify the unit of observation. Does each row represent an incident, a group of people, or another type of record?
2. List the five columns with the highest percentage of missing values.
3. Explain why a missing value is not equivalent to zero.
4. Select one numeric count variable and explain its mean, median, and spread.
5. Compare documented record counts across regions or routes. What other information would be needed to compare risk?
6. Explain why the year field must be analyzed as a year rather than converted directly from an integer into a timestamp.
7. Investigate an extreme numeric value and explain whether it is plausible.
8. If coordinates exist, report how many records have valid latitude and longitude.
9. Write a short data-quality note documenting cleaning choices and limitations.

## 21. Interpretation and ethics checklist

- The dataset is likely incomplete and subject to reporting and documentation biases.
- A higher number of records in one region does not automatically mean the underlying risk is higher; reporting coverage and route volume may differ.
- Missing values are not zeros.
- Correlation does not imply causation; some count variables may also be mathematically related.
- Check duplicate rows before removing them. Identical rows are not always proof of duplicate real-world events.
- Do not publish precise or potentially identifying location information about vulnerable people.
- Document filtering, cleaning, date parsing, and any assumptions.
- Interpret results as patterns in **documented records**, not as a complete census of migration-related deaths and disappearances.

## Key takeaway

Good data analysis combines appropriate data structures, careful cleaning, sound statistical summaries, clear visualizations, and responsible interpretation of uncertainty.
